# Prueba de progreso 2: Clustering de tráfico en Gipuzkoa

Este cuaderno sigue el enunciado de `index 2.pdf` y utiliza los tres ficheros proporcionados:
- `20251208_datosvelocidad(1).csv`
- `20251208_datosvolumen (1).csv`
- `estaciones(1).csv`

Objetivo: construir perfiles de tráfico por estación y hora, comparar **KMeans**, **DBSCAN** y **Agglomerative Clustering**, y seleccionar automáticamente el mejor modelo con métricas internas.

## 1) Configuración del entorno y librerías

Se importan librerías de análisis, visualización, preprocesado y clustering. Además, fijamos semilla para reproducibilidad.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

BASE_DIR = Path('.')
FILE_VEL = BASE_DIR / '20251208_datosvelocidad.csv'
FILE_VOL = BASE_DIR / '20251208_datosvolumen.csv'
FILE_EST = BASE_DIR / 'estaciones.csv'

print('Rutas detectadas:')
print(FILE_VEL.resolve())
print(FILE_VOL.resolve())
print(FILE_EST.resolve())

## 2) Carga de datos de velocidad, volumen y estaciones

Se leen los CSV con separador `;` y codificación compatible con acentos. Se limpian espacios extra en columnas y contenido.

In [ ]:
def read_semicolon_csv(path: Path) -> pd.DataFrame:
    # index_col=False evita que pandas use la primera columna como indice
    # cuando hay separador final sobrante en algunas filas.
    df = pd.read_csv(path, sep=';', encoding='latin-1', dtype=str, index_col=False)
    # Elimina columnas vacias creadas por separadores finales
    df = df.dropna(axis=1, how='all')
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({'': np.nan, 'nan': np.nan})
    # Fuerza indice incremental consistente entre datasets
    df = df.reset_index(drop=True)
    return df

vel = read_semicolon_csv(FILE_VEL)
vol = read_semicolon_csv(FILE_VOL)
est = read_semicolon_csv(FILE_EST)

print('Shapes:')
print('vel:', vel.shape)
print('vol:', vol.shape)
print('est:', est.shape)

print('\nDtypes (raw):')
print('vel\n', vel.dtypes.head())
print('vol\n', vol.dtypes.head())
print('est\n', est.dtypes.head())

print('\nMuestra velocidad:')
display(vel.head(3))
print('Muestra volumen:')
display(vol.head(3))
print('Muestra estaciones:')
display(est.head(3))

## 3) Limpieza, codificación y tipado de columnas

Se convierten columnas numéricas, se normalizan decimales con coma y se extrae el código de ETD para poder vincular los tres datasets.

In [ ]:
def to_numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s.astype(str).str.replace(',', '.', regex=False), errors='coerce')

# ---------- ESTACIONES ----------
est = est.rename(columns={
    'ETD code': 'etd_code',
    'Description': 'etd_description',
    'System': 'system'
})
est['etd_code'] = pd.to_numeric(est['etd_code'], errors='coerce').astype('Int64')
est['X'] = to_numeric_series(est['X'])
est['Y'] = to_numeric_series(est['Y'])

# ---------- VELOCIDAD ----------
vel = vel.rename(columns={
    'Fecha': 'fecha',
    'Hora': 'hora_rango',
    'Sistema': 'system',
    'ETD': 'etd_description',
    'Detector': 'detector',
    '0-50 (km/h)': 'v_0_50',
    '50-80 (km/h)': 'v_50_80',
    '80-120 (km/h)': 'v_80_120',
    '120-255 (km/h)': 'v_120_255',
    'Velocidad media (km/h)': 'vel_media',
    '0-6 (m)': 'dist_0_6',
    '6-999 (m)': 'dist_6_999',
    'Vehículos totales': 'veh_totales'
})

vel['etd_code'] = pd.to_numeric(
    vel['etd_description'].str.extract(r'\]\s*(\d+)-ETD', expand=False),
    errors='coerce'
).astype('Int64')

for c in ['v_0_50', 'v_50_80', 'v_80_120', 'v_120_255', 'vel_media', 'dist_0_6', 'dist_6_999', 'veh_totales']:
    vel[c] = to_numeric_series(vel[c])

# ---------- VOLUMEN ----------
vol = vol.rename(columns={
    'Estacion': 'estacion',
    'Fecha': 'fecha',
    'Hora': 'hora'
})
vol['etd_code'] = pd.to_numeric(vol['estacion'], errors='coerce').astype('Int64')

for c in vol.columns:
    if 'Carril' in c:
        vol[c] = pd.to_numeric(vol[c], errors='coerce')


## 4) Normalización temporal y unión de datasets

Se armoniza la granularidad temporal: velocidad (cada 30 min) se agrega a hora, y volumen (hora) se transforma a `datetime` consistente para fusionar por `etd_code` + hora.

In [ ]:
# --- Tiempo en volumen (manejo especial de 24:00) ---
vol['fecha_dt'] = pd.to_datetime(vol['fecha'], format='%d/%m/%Y', errors='coerce')
vol['hora_int'] = pd.to_numeric(vol['hora'].str.split(':').str[0], errors='coerce')
vol['datetime'] = vol['fecha_dt'] + pd.to_timedelta(vol['hora_int'].fillna(0), unit='h')
# 24:00 corresponde al inicio del día siguiente
mask_24 = vol['hora_int'] == 24
vol.loc[mask_24, 'datetime'] = vol.loc[mask_24, 'fecha_dt'] + pd.Timedelta(days=1)

# --- Tiempo en velocidad (rango de media hora -> inicio de intervalo) ---
vel['hora_inicio'] = vel['hora_rango'].str.split('-').str[0].str.strip()
vel['datetime_30m'] = pd.to_datetime(
    vel['fecha'] + ' ' + vel['hora_inicio'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)
vel['datetime'] = vel['datetime_30m'].dt.floor('h')

# --- Agregación velocidad a 1 hora por estación ---
def weighted_avg(x_vals, w_vals):
    w_sum = np.nansum(w_vals)
    if w_sum == 0:
        return np.nan
    return np.nansum(x_vals * w_vals) / w_sum

vel_hour = (
    vel.groupby(['etd_code', 'datetime'], dropna=False)
       .apply(lambda g: pd.Series({
           'vel_veh_totales': g['veh_totales'].sum(skipna=True),
           'vel_media_pond': weighted_avg(g['vel_media'].values, g['veh_totales'].values),
           'v_0_50': g['v_0_50'].sum(skipna=True),
           'v_50_80': g['v_50_80'].sum(skipna=True),
           'v_80_120': g['v_80_120'].sum(skipna=True),
           'v_120_255': g['v_120_255'].sum(skipna=True),
           'dist_0_6': g['dist_0_6'].sum(skipna=True),
           'dist_6_999': g['dist_6_999'].sum(skipna=True)
       }))
       .reset_index()
)

# --- Unión de fuentes ---
merged = vol.merge(vel_hour, on=['etd_code', 'datetime'], how='inner')
merged = merged.merge(
    est[['etd_code', 'system', 'etd_description', 'Territory', 'Municipality', 'X', 'Y']],
    on='etd_code',
    how='left'
)

print('Registros tras unión:', merged.shape)
print('Rango temporal:', merged['datetime'].min(), '->', merged['datetime'].max())
display(merged.head(3))

## 5) Ingeniería de variables para clustering

Se generan variables de intensidad y composición del tráfico (ligeros/pesados), distribución por rangos de velocidad y componentes temporales (hora y día).

In [ ]:
# Columnas por carril
lig_cols = [c for c in merged.columns if 'ligeros' in c.lower()]
pes_cols = [c for c in merged.columns if 'pesados' in c.lower()]

for c in lig_cols + pes_cols:
    merged[c] = pd.to_numeric(merged[c], errors='coerce')

merged['vol_ligeros_total'] = merged[lig_cols].sum(axis=1, skipna=True)
merged['vol_pesados_total'] = merged[pes_cols].sum(axis=1, skipna=True)
merged['vol_total'] = merged['vol_ligeros_total'] + merged['vol_pesados_total']

merged['pct_pesados'] = np.where(
    merged['vol_total'] > 0,
    100 * merged['vol_pesados_total'] / merged['vol_total'],
    np.nan
)

activos = ((merged[lig_cols + pes_cols].fillna(0) > 0).sum(axis=1)).replace(0, np.nan)
merged['intensidad_media_carril'] = merged['vol_total'] / activos

# Porcentajes de rangos de velocidad
merged['speed_bins_total'] = merged[['v_0_50', 'v_50_80', 'v_80_120', 'v_120_255']].sum(axis=1, skipna=True)
for c in ['v_0_50', 'v_50_80', 'v_80_120', 'v_120_255']:
    merged[f'pct_{c}'] = np.where(
        merged['speed_bins_total'] > 0,
        100 * merged[c] / merged['speed_bins_total'],
        np.nan
    )

merged['pct_dist_0_6'] = np.where(
    (merged['dist_0_6'] + merged['dist_6_999']) > 0,
    100 * merged['dist_0_6'] / (merged['dist_0_6'] + merged['dist_6_999']),
    np.nan
)

merged['hora'] = merged['datetime'].dt.hour
merged['dia_semana'] = merged['datetime'].dt.dayofweek
merged['es_fin_de_semana'] = (merged['dia_semana'] >= 5).astype(int)

print('Variables derivadas creadas.')
display(merged[['etd_code', 'datetime', 'vol_total', 'pct_pesados', 'vel_media_pond', 'pct_v_0_50', 'pct_v_80_120']].head(5))

## 6) Preprocesado numerico y escalado

Se seleccionan variables finales, se imputan faltantes con mediana y se escalan con `MinMaxScaler` para que todos los algoritmos trabajen en la misma base.

In [ ]:
features = [
    'vol_total', 'vol_ligeros_total', 'vol_pesados_total', 'pct_pesados', 'intensidad_media_carril',
    'vel_veh_totales', 'vel_media_pond', 'pct_v_0_50', 'pct_v_50_80', 'pct_v_80_120', 'pct_v_120_255',
    'pct_dist_0_6', 'hora', 'dia_semana', 'es_fin_de_semana', 'X', 'Y'
]

df_model = merged.copy()
X_raw = df_model[features]

preprocess = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

X = preprocess.fit_transform(X_raw)

print('Matriz de entrenamiento:', X.shape)
print('Numero de estaciones distintas:', df_model['etd_code'].nunique())

## 7) Entrenamiento simultáneo de KMeans, DBSCAN y Agglomerative

Se lanzan los tres algoritmos con parámetros base sobre el mismo dataset preprocesado para una primera comparación.

In [ ]:
from sklearn.neighbors import NearestNeighbors


def suggest_dbscan_eps(X_data, min_samples=10, quantile=0.9):
    """Estima un eps razonable con la distancia al k-esimo vecino."""
    nn = NearestNeighbors(n_neighbors=min_samples, metric='euclidean')
    nn.fit(X_data)
    distances, _ = nn.kneighbors(X_data)
    k_dist = distances[:, -1]
    return float(np.quantile(k_dist, quantile))


def eval_cluster_labels(X_data, labels, max_eval_samples=5000):
    labels = np.array(labels)
    mask = labels != -1
    n_clusters_total = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    noise_ratio = n_noise / len(labels)

    if n_clusters_total >= 2 and mask.sum() >= 3 and len(np.unique(labels[mask])) >= 2:
        # Para evitar bloqueos con datasets grandes, evaluamos sobre una muestra.
        if mask.sum() > max_eval_samples:
            rng = np.random.default_rng(SEED)
            idx = rng.choice(np.where(mask)[0], size=max_eval_samples, replace=False)
            X_eval = X_data[idx]
            y_eval = labels[idx]
        else:
            X_eval = X_data[mask]
            y_eval = labels[mask]

        sil = silhouette_score(X_eval, y_eval)
        cal = calinski_harabasz_score(X_eval, y_eval)
        dav = davies_bouldin_score(X_eval, y_eval)
    else:
        sil, cal, dav = np.nan, np.nan, np.nan

    return {
        'n_clusters': n_clusters_total,
        'n_noise': n_noise,
        'noise_ratio': noise_ratio,
        'silhouette': sil,
        'calinski_harabasz': cal,
        'davies_bouldin': dav,
    }


base_eps = suggest_dbscan_eps(X, min_samples=10, quantile=0.9)
print(f'eps base sugerido para DBSCAN (MinMax): {base_eps:.4f}')

base_models = {
    'KMeans': KMeans(n_clusters=4, random_state=SEED, n_init=20),
    'Agglomerative': AgglomerativeClustering(n_clusters=4, linkage='ward'),
    'DBSCAN': DBSCAN(eps=base_eps, min_samples=10)
}

base_rows = []
base_labels = {}

for name, model in base_models.items():
    labels = model.fit_predict(X)
    base_labels[name] = labels
    m = eval_cluster_labels(X, labels)
    m['modelo'] = name
    m['params'] = str(model.get_params())
    base_rows.append(m)

base_results = pd.DataFrame(base_rows).sort_values('silhouette', ascending=False)
display(base_results)

## 8) Búsqueda de hiperparámetros y ejecución en bucle

Se prueba una rejilla sencilla de parámetros para cada algoritmo y se guardan todas las corridas en una tabla única para comparar.

In [ ]:
search_rows = []

# KMeans
for k in range(3, 8):
    model = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    labels = model.fit_predict(X)
    m = eval_cluster_labels(X, labels)
    m.update({'modelo': 'KMeans', 'params': {'n_clusters': k}, 'labels': labels})
    search_rows.append(m)

# Agglomerative
for k in range(3, 8):
    for linkage in ['ward', 'complete', 'average']:
        model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        labels = model.fit_predict(X)
        m = eval_cluster_labels(X, labels)
        m.update({'modelo': 'Agglomerative', 'params': {'n_clusters': k, 'linkage': linkage}, 'labels': labels})
        search_rows.append(m)

# DBSCAN: rejilla centrada en eps sugerido por datos
# Factores amplios para cubrir zonas con estructura y no solo ruido.
eps_factors = [0.4, 0.6, 0.8, 1.0, 1.2, 1.4]
for min_samples in [8, 12, 16]:
    eps_center = suggest_dbscan_eps(X, min_samples=min_samples, quantile=0.9)
    for f in eps_factors:
        eps = float(np.round(eps_center * f, 4))
        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels = model.fit_predict(X)
        m = eval_cluster_labels(X, labels)
        m.update({'modelo': 'DBSCAN', 'params': {'eps': eps, 'min_samples': min_samples}, 'labels': labels})
        search_rows.append(m)

results = pd.DataFrame(search_rows)
print('Total de corridas:', len(results))
print('Corridas con metricas validas:', results['silhouette'].notna().sum())
print('Corridas validas DBSCAN:', results.query("modelo == 'DBSCAN' and silhouette == silhouette").shape[0])
display(results.head(10))

## 9) Evaluacion con silhouette (criterio principal)

Se selecciona el mejor modelo usando solo `silhouette`, aplicando filtros de calidad para evitar soluciones con demasiado ruido o clusters demasiado pequenos.

In [ ]:
# Nos quedamos con corridas donde silhouette esta definida
rank_df = results[results['silhouette'].notna()].copy()

# Filtro de calidad para evitar soluciones degeneradas.
# 1) Al menos 2 clusters reales.
# 2) En DBSCAN, acotamos ruido maximo.
rank_df = rank_df[
    (rank_df['n_clusters'] >= 2) &
    ((rank_df['modelo'] != 'DBSCAN') | (rank_df['noise_ratio'] <= 0.2))
].copy()

# 3) Tamano minimo del cluster mas pequeno (sin contar ruido en DBSCAN).
def min_cluster_size(labels):
    arr = np.array(labels)
    arr = arr[arr != -1]
    if arr.size == 0:
        return 0
    return int(pd.Series(arr).value_counts().min())

rank_df['min_cluster_size'] = rank_df['labels'].apply(min_cluster_size)
min_cluster_abs = max(20, int(0.005 * len(X)))
rank_df = rank_df[rank_df['min_cluster_size'] >= min_cluster_abs].copy()

if rank_df.empty:
    raise ValueError('No hay corridas validas para evaluar con los filtros de silhouette.')

# Criterio principal: silhouette (desc), y desempate por menos ruido.
rank_df['silhouette'] = pd.to_numeric(rank_df['silhouette'], errors='coerce')
rank_df['noise_ratio'] = pd.to_numeric(rank_df['noise_ratio'], errors='coerce')
rank_df = rank_df.sort_values(['silhouette', 'noise_ratio'], ascending=[False, True]).reset_index(drop=True)

# Conservamos score_global por compatibilidad con celdas posteriores.
rank_df['score_global'] = rank_df['silhouette']

cols_show = ['modelo', 'params', 'n_clusters', 'min_cluster_size', 'noise_ratio', 'silhouette', 'score_global']
display(rank_df[cols_show].head(15))

## 10) Visualizacion de clusters y comparacion final

Se proyecta a 2 dimensiones con PCA para comparar visualmente los clusters de la mejor configuracion de cada algoritmo segun `silhouette`.

In [ ]:
# Seleccion por algoritmo evitando, si es posible, soluciones casi identicas.
algo_order = ['KMeans', 'Agglomerative', 'DBSCAN']
selected_rows = []
selected_label_sets = []

for algo in algo_order:
    candidates = rank_df[rank_df['modelo'] == algo].copy()
    if candidates.empty:
        continue

    chosen = None
    for _, row in candidates.iterrows():
        labels_row = np.array(row['labels'])
        if not selected_label_sets:
            chosen = row
            break

        max_ari = max(adjusted_rand_score(labels_row, prev_labels) for prev_labels in selected_label_sets)
        if max_ari < 0.999:
            chosen = row
            break

    if chosen is None:
        # Si no hay alternativa distinta, se toma la mejor por silhouette.
        chosen = candidates.iloc[0]

    selected_rows.append(chosen)
    selected_label_sets.append(np.array(chosen['labels']))

best_by_algo = pd.DataFrame(selected_rows).reset_index(drop=True)
display(best_by_algo[['modelo', 'params', 'silhouette', 'noise_ratio', 'min_cluster_size']])

# PCA para visualizacion
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharex=True, sharey=True)
for ax, algo in zip(axes, algo_order):
    subset = best_by_algo[best_by_algo['modelo'] == algo]

    if subset.empty:
        ax.set_title(f"{algo} | sin configuracion valida")
        ax.text(0.5, 0.5, 'No disponible en ranking', ha='center', va='center', transform=ax.transAxes)
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        continue

    row = subset.iloc[0]
    params = row['params']

    if algo == 'KMeans':
        model = KMeans(n_clusters=params['n_clusters'], random_state=SEED, n_init=20)
    elif algo == 'Agglomerative':
        model = AgglomerativeClustering(n_clusters=params['n_clusters'], linkage=params['linkage'])
    else:
        model = DBSCAN(eps=params['eps'], min_samples=params['min_samples'])

    labels = model.fit_predict(X)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=12, alpha=0.8)
    ax.set_title(f"{algo} | silhouette={row['silhouette']:.2f}")
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('Comparacion visual de clusters (PCA 2D)', y=1.03)
plt.tight_layout()
plt.show()

# Ranking de silhouette por modelo (mejor configuracion)
plt.figure(figsize=(8, 4))
sns.barplot(data=best_by_algo, x='modelo', y='silhouette', palette='Set2')
plt.title('Silhouette (mejor configuracion por algoritmo)')
plt.ylabel('Silhouette')
plt.xlabel('Algoritmo')
plt.show()

## 11) Seleccion del mejor modelo y exportacion de resultados

Se selecciona automaticamente la mejor corrida por `silhouette` (con filtros de calidad), se asigna la etiqueta de cluster al dataset y se exportan resultados y metricas a CSV.

In [ ]:
best = rank_df.iloc[0]
print('Mejor configuracion encontrada (criterio: silhouette):')
display(best[['modelo', 'params', 'n_clusters', 'min_cluster_size', 'noise_ratio', 'silhouette']])

if best['modelo'] == 'KMeans':
    final_model = KMeans(n_clusters=best['params']['n_clusters'], random_state=SEED, n_init=20)
elif best['modelo'] == 'Agglomerative':
    final_model = AgglomerativeClustering(
        n_clusters=best['params']['n_clusters'],
        linkage=best['params']['linkage']
    )
else:
    final_model = DBSCAN(
        eps=best['params']['eps'],
        min_samples=best['params']['min_samples']
    )

final_labels = final_model.fit_predict(X)

resultados = df_model.copy()
resultados['cluster'] = final_labels
resultados['modelo_ganador'] = best['modelo']

metricas_export = rank_df.drop(columns=['labels']).copy()

out_resultados = BASE_DIR / 'resultados_clustering_trafico.csv'
out_metricas = BASE_DIR / 'metricas_clustering_trafico.csv'

resultados.to_csv(out_resultados, index=False, encoding='utf-8-sig')
metricas_export.to_csv(out_metricas, index=False, encoding='utf-8-sig')

print('Archivos exportados:')
print(out_resultados.resolve())
print(out_metricas.resolve())

print('\nResumen por cluster (media de variables clave):')
summary_cols = ['vol_total', 'pct_pesados', 'vel_media_pond', 'pct_v_0_50', 'pct_v_80_120', 'hora']
display(resultados.groupby('cluster')[summary_cols].mean().round(2).sort_index())